# CRISPRtracrRNA Prediction

![CRISPRtracrRNA Prediction](https://proto-bio.github.io/proto-assets/images/tool/crispr_tracr_rna/hero.png)

This notebook demonstrates `run_crispr_tracr_rna`, which scans nucleotide sequences for tracrRNA candidates using the CRISPRtracrRNA tool from the Backofen Lab. tracrRNA is required by type II CRISPR-Cas9 systems to form the guide RNA complex; identifying it is a critical step when characterizing novel Cas9 orthologs or validating candidate loci in a CRISPR engineering pipeline.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("crispr_tracr_rna")
display_overview("crispr_tracr_rna")
display_docs_section("crispr_tracr_rna", "Background")

# CRISPRtracrRNA

[CRISPRtracrRNA](https://github.com/BackofenLab/CRISPRtracrRNA) is a multi-evidence pipeline from the [Bioinformatics Group at the University of Freiburg](https://www.bioinf.uni-freiburg.de/) that detects [tracrRNA](https://en.wikipedia.org/wiki/Trans-activating_crRNA) candidates in nucleotide [CRISPR](https://en.wikipedia.org/wiki/CRISPR) loci. It combines covariance-model search, CRISPR array detection, Cas-effector cassette detection, anti-repeat similarity, RNA-RNA interaction prediction, and transcription-terminator detection into a single weighted ranking score per candidate.

[CRISPRtracrRNA](https://github.com/BackofenLab/CRISPRtracrRNA) ([Mitrofanov et al., 2022](https://doi.org/10.1093/bioinformatics/btac466)) detects trans-activating CRISPR RNA (tracrRNA) sequences in nucleotide CRISPR loci. A tracrRNA is a small non-coding RNA that base-pairs with the precursor crRNA in the [Class 2](https://en.wikipedia.org/wiki/CRISPR) effector systems that depend on one, namely [Type II](https://en.wikipedia.org/wiki/Cas9) (Cas9) and the tracrRNA-bearing [Type V](https://en.wikipedia.org/wiki/Cas12a) subtypes such as Cas12b, Cas12c, and Cas12e, and the resulting RNA duplex licenses Cas-mediated cleavage of target DNA. It is also the second component fused into the [single-guide RNA](https://en.wikipedia.org/wiki/Guide_RNA) used in modern genome editing. Because tracrRNAs share little primary-sequence conservation across families, single-model approaches such as [Infernal](https://github.com/EddyRivasLab/infernal) covariance-model search alone miss divergent tracrRNAs in newly sequenced and metagenomic genomes.

Internally, the pipeline runs an array-detection step with [CRISPRidentify](https://github.com/BackofenLab/CRISPRidentify) (machine learning), a Cas-cassette step with [CRISPRcasIdentifier](https://github.com/BackofenLab/CRISPRcasIdentifier) (HMM and machine learning), a tracrRNA candidate scan with Infernal `cmsearch` against curated covariance models, an anti-repeat alignment step using fasta36, vmatch, Clustal Omega, and BLAST, an RNA-RNA interaction step with [IntaRNA](https://github.com/BackofenLab/IntaRNA), and a transcription-terminator step with erpin. A final ranking step combines the per-candidate features into a single weighted score, and a faster `model_run` mode performs only the covariance-model scan and skips the validation evidence and the ranking step.

## Available tools

In [2]:
display_available_tools("crispr_tracr_rna")

- **`run_crispr_tracr_rna()`** — Predict tracrRNA sequences from nucleotide CRISPR loci

### `run_crispr_tracr_rna`

`run_crispr_tracr_rna` predicts tracrRNA sequences from nucleotide sequences that contain CRISPR loci. The tool evaluates anti-repeat complementarity and uses IntaRNA to compute RNA-RNA interaction energies, returning predicted tracrRNA coordinates and interaction energy scores for each input sequence. In practice the input should be a confirmed CRISPR-containing genomic region, typically identified by a tool like MinCED in an earlier analysis step.

In [3]:
from proto_tools import (
    CrisprTracrRNAInput,
    CrisprTracrRNAConfig,
    run_crispr_tracr_rna,
)

In [4]:
# Display input docs
display_api_reference("crispr_tracr_rna", "input", "run_crispr_tracr_rna")

from pathlib import Path

# Real Streptococcus pyogenes SF370 CRISPR1-Cas9 locus (NC_002737.2).
locus = "".join(l for l in Path("spyogenes_crispr_locus.fasta").read_text().splitlines() if not l.startswith(">"))
inputs = CrisprTracrRNAInput(sequences=[locus])

**Input** — `CrisprTracrRNAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Nucleotide sequence(s) to predict tracrRNA from |

In [5]:
# Display config docs
display_api_reference("crispr_tracr_rna", "config", "run_crispr_tracr_rna")

# Type II model for Cas9-associated tracrRNA prediction | see docs above for all fields
config = CrisprTracrRNAConfig(model_type="II")

**Config** — `CrisprTracrRNAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_type</code> | <code>Literal['II', 'all']</code> | <code>'II'</code> | CRISPR model: 'II' (type II only, fast, default) or 'all' (type II + type V cluster models) |
| <code>run_type</code> | <code>Literal['complete_run', 'model_run']</code> | <code>'complete_run'</code> | Pipeline mode: 'complete_run' (full ranking, default) or 'model_run' (cmsearch scan only, fast) |
| <code>num_workers</code> | <code>int &#124; None</code> | <code>None</code> | Parallel workers across input sequences. None or 0 defaults to 1; capped at len(sequences). |
| <code>anti_repeat_similarity_threshold</code> | <code>float</code> | <code>0.7</code> | Anti-repeat ↔ repeat similarity floor (0-1). Lower for divergent CRISPR families |
| <code>anti_repeat_coverage_threshold</code> | <code>float</code> | <code>0.6</code> | Anti-repeat alignment coverage floor (0-1). Lower for partial anti-repeats |
| <code>weight_crispr_array_score</code> | <code>float</code> | <code>0.5</code> | Multi-evidence ranking weight for CRISPRidentify array-detection confidence. |
| <code>weight_anti_repeat_sim</code> | <code>float</code> | <code>0.5</code> | Multi-evidence ranking weight for anti-repeat sequence similarity. |
| <code>weight_anti_repeat_coverage</code> | <code>float</code> | <code>0.5</code> | Multi-evidence ranking weight for anti-repeat alignment coverage. |
| <code>weight_anti_sim_coverage</code> | <code>float</code> | <code>0.5</code> | Multi-evidence ranking weight for the similarity x coverage product. |
| <code>weight_interaction_score</code> | <code>float</code> | <code>0.6</code> | Multi-evidence ranking weight for the IntaRNA RNA-RNA interaction energy. |
| <code>weight_model_hit_score</code> | <code>float</code> | <code>0.9</code> | Multi-evidence ranking weight for the covariance-model tail hit score. |
| <code>weight_terminator_hit_score</code> | <code>float</code> | <code>0.9</code> | Multi-evidence ranking weight for erpin terminator presence/score. |
| <code>weight_consistency_orientation</code> | <code>float</code> | <code>0.1</code> | Multi-evidence ranking weight for repeat / anti-repeat orientation consistency. |
| <code>weight_consistency_anti_repeat_tail</code> | <code>float</code> | <code>0.1</code> | Multi-evidence ranking weight for anti-repeat ↔ tail positional consistency. |
| <code>weight_consistency_tail_terminator</code> | <code>float</code> | <code>0.1</code> | Multi-evidence ranking weight for tail ↔ terminator positional consistency. |
| <code>perform_type_v_anti_repeat_analysis</code> | <code>bool</code> | <code>False</code> | Search Type V (Cas12) anti-repeat locations. Niche; off by default |

In [6]:
# Run the tracrRNA prediction
result = run_crispr_tracr_rna(inputs, config)

Running run_crispr_tracr_rna [00:00]

In [7]:
# Display output docs
display_api_reference("crispr_tracr_rna", "output", "run_crispr_tracr_rna")

# One CrisprTracrRNASequenceResult per input sequence; each carries every candidate hit
# upstream produced for that accession, sorted by score descending.
for seq_result in result.results:
    top = seq_result.top_candidate
    if top is not None and top.has_tracr:
        print(f"{seq_result.sequence_id}: tracrRNA at {top.anti_repeat_start}-{top.anti_repeat_end} "
              f"(score={top.score}, {len(seq_result.candidates)} candidates)")
        print(f"  Interaction energy: {top.interaction_energy} kcal/mol")
    else:
        print(f"{seq_result.sequence_id}: no tracrRNA detected")

**Output** — `CrisprTracrRNAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[CrisprTracrRNASequenceResult]</code> | <code>[]</code> | Per-input sequence results, each with all candidate hits top-ranked first. |

**`CrisprTracrRNASequenceResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_id</code> | <code>str</code> | required | ID of the input sequence |
| <code>candidates</code> | <code>list[CrisprTracrRNAPrediction]</code> | <code>[]</code> | All candidate hits for this sequence, top-ranked first. |

**`CrisprTracrRNAPrediction`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_id</code> | <code>str</code> | required | ID of the input sequence |
| <code>accession_number</code> | <code>str &#124; None</code> | <code>None</code> | Upstream's accession_number column. |
| <code>crispr_array_index</code> | <code>int &#124; None</code> | <code>None</code> | Index of the matched CRISPR array. |
| <code>crispr_array_category</code> | <code>str &#124; None</code> | <code>None</code> | CRISPR array category from CRISPRidentify. |
| <code>crispr_array_score</code> | <code>float &#124; None</code> | <code>None</code> | CRISPRidentify array-detection confidence. |
| <code>crispr_array_start</code> | <code>int &#124; None</code> | <code>None</code> | Start position of the CRISPR array. |
| <code>crispr_array_end</code> | <code>int &#124; None</code> | <code>None</code> | End position of the CRISPR array. |
| <code>crispr_array_repeat_consensus</code> | <code>str &#124; None</code> | <code>None</code> | Consensus repeat sequence. |
| <code>crispr_array_orientation</code> | <code>str &#124; None</code> | <code>None</code> | Predicted array orientation. |
| <code>crispr_orientation_flag</code> | <code>str &#124; None</code> | <code>None</code> | Orientation-confidence flag. |
| <code>anti_repeat_sequence</code> | <code>str &#124; None</code> | <code>None</code> | Anti-repeat sequence. |
| <code>anti_repeat_start</code> | <code>int &#124; None</code> | <code>None</code> | Anti-repeat start position. |
| <code>anti_repeat_end</code> | <code>int &#124; None</code> | <code>None</code> | Anti-repeat end position. |
| <code>anti_repeat_direction</code> | <code>str &#124; None</code> | <code>None</code> | Anti-repeat strand/direction. |
| <code>anti_repeat_relative_location</code> | <code>str &#124; None</code> | <code>None</code> | Anti-repeat location relative to the CRISPR array. |
| <code>anti_repeat_distance_from_crispr_array</code> | <code>int &#124; None</code> | <code>None</code> | Distance from the anti-repeat to the CRISPR array. |
| <code>anti_repeat_similarity</code> | <code>float &#124; None</code> | <code>None</code> | Anti-repeat sequence similarity (0-1). |
| <code>anti_repeat_coverage</code> | <code>float &#124; None</code> | <code>None</code> | Anti-repeat alignment coverage (0-1). |
| <code>anti_repeat_similarity_coverage_multiplication</code> | <code>float &#124; None</code> | <code>None</code> | Similarity multiplied by coverage. |
| <code>anti_repeat_upstream</code> | <code>str &#124; None</code> | <code>None</code> | Sequence upstream of the anti-repeat. |
| <code>tracr_rna_taken_flag</code> | <code>str &#124; None</code> | <code>None</code> | Flag for whether the tracrRNA was selected. |
| <code>tracr_rna_tail_sequence</code> | <code>str &#124; None</code> | <code>None</code> | 3' tail sequence of the tracrRNA. |
| <code>tracr_rna_global_window_sequence</code> | <code>str &#124; None</code> | <code>None</code> | Global window sequence around the tracrRNA. |
| <code>tracr_rna_sequence</code> | <code>str &#124; None</code> | <code>None</code> | Predicted tracrRNA sequence. |
| <code>intarna_anti_repeat_interaction_interval</code> | <code>str &#124; None</code> | <code>None</code> | Interval of the predicted RNA-RNA interaction. |
| <code>intarna_anti_repeat_interaction</code> | <code>str &#124; None</code> | <code>None</code> | IntaRNA anti-repeat interaction structure. |
| <code>interaction_energy</code> | <code>float &#124; None</code> | <code>None</code> | IntaRNA interaction energy in kcal/mol; more negative values indicate a stronger interaction. |
| <code>poli_u_signal_coordinates</code> | <code>str &#124; None</code> | <code>None</code> | Coordinates of the poly-U transcription terminator signal. |
| <code>terminator_all_locations</code> | <code>str &#124; None</code> | <code>None</code> | All erpin terminator hit locations. |
| <code>terminator_all_scores</code> | <code>str &#124; None</code> | <code>None</code> | All erpin terminator hit scores. |
| <code>best_terminator_location</code> | <code>str &#124; None</code> | <code>None</code> | Best erpin terminator location. |
| <code>best_terminator_score</code> | <code>float &#124; None</code> | <code>None</code> | Best erpin terminator score. |
| <code>terminator_presence_flag</code> | <code>str &#124; None</code> | <code>None</code> | Flag for terminator presence. |
| <code>tail_model_hit_location</code> | <code>str &#124; None</code> | <code>None</code> | Tail model hit location. |
| <code>tail_model_hit_score</code> | <code>float &#124; None</code> | <code>None</code> | Tail model hit score. |
| <code>tail_presence_flag</code> | <code>str &#124; None</code> | <code>None</code> | Flag for tail presence. |
| <code>closest_corresponding_cas_interval</code> | <code>str &#124; None</code> | <code>None</code> | Coordinates of the closest cas-effector cassette. |
| <code>distance_to_cas</code> | <code>int &#124; None</code> | <code>None</code> | Distance from the tracrRNA to the cas cassette. |
| <code>score</code> | <code>float &#124; None</code> | <code>None</code> | Multi-evidence ranking score (weighted sum across all evidence fields). |
| <code>start</code> | <code>int &#124; None</code> | <code>None</code> | cmsearch hit start position (model_run mode). |
| <code>end</code> | <code>int &#124; None</code> | <code>None</code> | cmsearch hit end position (model_run mode). |
| <code>e_value</code> | <code>float &#124; None</code> | <code>None</code> | cmsearch hit e-value (model_run mode). |
| <code>best_e_value</code> | <code>float &#124; None</code> | <code>None</code> | cmsearch best-hit e-value (model_run mode). |
| <code>hit_sequence</code> | <code>str &#124; None</code> | <code>None</code> | cmsearch hit sequence (model_run mode). |

seq_0: tracrRNA at 1295-1322 (score=1.9504838420832522, 4 candidates)
  Interaction energy: -6.57 kcal/mol


## Export Results

TracrRNA predictions can be exported to CSV or JSON. Both formats flatten to one row per candidate hit — sequences with multiple candidates produce multiple rows (sorted by `score` descending), and sequences with no candidates produce a single `sequence_id`-only row.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispr_tracr_rna_results", export_path=output_dir, file_format="json")
print(f"tracrRNA predictions exported to {output_dir}")

tracrRNA predictions exported to example_output
